# 05 · Sanity Check — Agente Crítico

**Papel deste notebook:** fazer o que um revisor cético faria antes de assinar embaixo do
diagnóstico — recomputar os números mais citados **do zero, a partir da base bruta**
(`../Dados do Case/*.csv`), com código escrito de forma independente, e confrontar cada
resultado contra o que os notebooks publicaram: os **números canônicos**
(`06` → `outputs/numeros_canonicos.json`) e a **avaliação de impacto**
(`07` → `outputs/impacto.json`).

**Regra deste notebook:** nenhuma célula reaproveita função ou variável dos notebooks 06/07.
Tudo é recomputado aqui, por caminho de código próprio. É a independência que dá valor ao check: um número
conferido pelo mesmo código que o produziu não foi conferido.

**Cobertura:** além dos números canônicos e dos R$ de impacto, este notebook audita também a
camada de priorização (`08` → `outputs/priorizacao.json`) e verifica que as dashboards não
citam arquivo inexistente — foi assim que referências a arquivos-fantasma passaram batido antes.


In [1]:
import pandas as pd
import numpy as np
import json
from pathlib import Path
from scipy import stats

RAW = Path('..') / 'Dados do Case'
def carregar(n): return pd.read_csv(RAW / f'[BootCamp EloGroup 2026] {n}.csv', encoding='utf-8-sig')

# --- números publicados que vamos auditar ---
CANON = json.loads(Path('outputs/numeros_canonicos.json').read_text(encoding='utf-8'))['metricas']
IMP   = json.loads(Path('outputs/impacto.json').read_text(encoding='utf-8'))
def pub(chave): return CANON[chave]['valor']
def imp(iid):   return next(i for i in IMP['impactos'] if i['id']==iid)

resultados = []
def checar(nome, calculado, publicado, tol, fonte):
    ok = abs(calculado - publicado) <= tol
    resultados.append({'métrica': nome, 'recalculado': round(calculado,2), 'publicado': round(publicado,2),
                       'confere': 'SIM' if ok else 'NÃO — DIVERGÊNCIA', 'fonte': fonte})
    return ok

## Bloco A — Integridade da base bruta

Confirma que a base tem o tamanho esperado, chaves não duplicam, e as três fórmulas
financeiras de `vendas.csv` fecham 100%.

In [2]:
vendas = carregar('Vendas'); atend = carregar('Atendimento')
estoque = carregar('Estoque'); clientes = carregar('Clientes')
for nome, df, pk in [('vendas',vendas,'order_id'),('atendimento',atend,'ticket_id'),
                     ('estoque',estoque,'sku_id'),('clientes',clientes,'customer_id')]:
    print(f"{nome:<12} linhas={len(df):<7} dup_chave={df[pk].duplicated().sum()}")

vf = vendas.dropna(subset=['quantidade']).copy()
c1 = ((vf['preco_unitario']*vf['quantidade'] - vf['receita_bruta']).abs() > 0.01).sum()
c2 = ((vf['receita_bruta'] - vf['desconto_reais'] - vf['receita_liquida']).abs() > 0.01).sum()
c3 = ((vf['receita_liquida'] - vf['custo_produto'] - vf['custo_frete'] - vf['margem_contribuicao']).abs() > 0.01).sum()
print(f"\nFórmulas financeiras com divergência >R$0,01: bruta={c1} liquida={c2} margem={c3} (esperado 0/0/0)")
checar('Pedidos válidos', len(vf), pub('pedidos_validos'), 0, 'nb06')

vendas       linhas=27759   dup_chave=0
atendimento  linhas=35841   dup_chave=0
estoque      linhas=5000    dup_chave=0
clientes     linhas=15000   dup_chave=0

Fórmulas financeiras com divergência >R$0,01: bruta=0 liquida=0 margem=0 (esperado 0/0/0)


True

## Bloco B — Recomputação independente dos números canônicos (nb06)

Cada valor abaixo é escrito do zero e comparado ao `numeros_canonicos.json`.

In [3]:
v = vendas.dropna(subset=['quantidade']).copy()
v['data_pedido'] = pd.to_datetime(v['data_pedido'])
v['devolvido'] = v['devolvido'].astype(bool)
rl_tot = v['receita_liquida'].sum(); rb_tot = v['receita_bruta'].sum(); mc_tot = v['margem_contribuicao'].sum()
dias = (v['data_pedido'].max() - v['data_pedido'].min()).days
anu = 365.0/dias

# margem contábil e realizada
checar('Margem contábil %', 100*mc_tot/rl_tot, pub('margem_contabil_pct'), 0.02, 'nb06')
realiz = v[(~v['devolvido']) & (v['status_pagamento']=='Aprovado')]
checar('Margem realizada %', 100*realiz['margem_contribuicao'].sum()/rl_tot, pub('margem_realizada_pct'), 0.02, 'nb06')
gap13 = mc_tot - realiz['margem_contribuicao'].sum()
checar('Gap de margem 13m (R$)', gap13, pub('margem_gap_rs_13m'), 1.0, 'nb06')

# desconto
checar('Desconto total 13m (R$)', v['desconto_reais'].sum(), pub('desconto_total_rs_13m'), 1.0, 'nb06')
checar('Desconto % receita bruta', 100*v['desconto_reais'].sum()/rb_tot, pub('desconto_pct_receita_bruta'), 0.02, 'nb06')

# marketplace (agregado, recomputado à mão)
mk = v[v['canal']=='Marketplace']; ot = v[v['canal']!='Marketplace']
gap_pp = 100*ot['margem_contribuicao'].sum()/ot['receita_liquida'].sum() - 100*mk['margem_contribuicao'].sum()/mk['receita_liquida'].sum()
checar('Marketplace gap (R$ 13m)', gap_pp/100*mk['receita_liquida'].sum(), pub('mkt_gap_rs_13m'), 1.0, 'nb06')
checar('Marketplace frete % agregado', 100*mk['custo_frete'].sum()/mk['receita_liquida'].sum(), pub('mkt_frete_pct_agregado'), 0.02, 'nb06')

# effect size desconto
vv = v.copy(); vv['dp'] = np.where(vv['receita_bruta']>0, vv['desconto_reais']/vv['receita_bruta'], np.nan)
vv['mp'] = vv['margem_contribuicao']/vv['receita_liquida']
dd = vv.dropna(subset=['dp','mp'])
checar('r² desconto×margem', stats.pearsonr(dd['dp'],dd['mp'])[0]**2, pub('es_desconto_r2'), 0.005, 'nb06')
pd.DataFrame(resultados).tail(9)

,métrica,recalculado,publicado,confere,fonte
0,Pedidos válidos,27758.00,27758.00,SIM,nb06
1,Margem contábil %,54.37,54.37,SIM,nb06
2,Margem realizada %,40.80,40.80,SIM,nb06
3,Gap de margem 13m (R$),2563716.19,2563716.19,SIM,nb06
4,Desconto total 13m (R$),1636799.83,1636799.83,SIM,nb06
5,Desconto % receita bruta,7.97,7.97,SIM,nb06
6,Marketplace gap (R$ 13m),148967.10,148967.10,SIM,nb06
7,Marketplace frete % agregado,4.93,4.93,SIM,nb06
8,r² desconto×margem,0.29,0.29,SIM,nb06


## Bloco C — Recomputação independente dos R$ de impacto (nb07)

A parte que a versão anterior deste notebook **não** auditava. Recomputa cada frente da
régua, anualiza pelo mesmo critério (vendas ×365/dias; atendimento ÷3) e compara ao
`impacto.json`.

In [4]:
# margem não realizada anualizada
checar('Impacto: margem não realizada (R$/ano)', gap13*anu, imp('margem_nao_realizada')['rs_ano'], 1.0, 'nb07')
# desconto anualizado
checar('Impacto: desconto (R$/ano)', v['desconto_reais'].sum()*anu, imp('desconto')['rs_ano'], 1.0, 'nb07')
# marketplace anualizado
checar('Impacto: marketplace frete (R$/ano)', (gap_pp/100*mk['receita_liquida'].sum())*anu, imp('marketplace_frete')['rs_ano'], 1.0, 'nb07')

# atendimento (36m)
a = atend.dropna(subset=['customer_id']).copy()
falha = ['Onde está meu pedido?','Defeito','Pagamento não aprovado']
checar('Impacto: custo falha operacional (R$/ano)',
       a.loc[a['categoria_problema'].isin(falha),'custo_operacional_ticket'].sum()/3,
       imp('falha_operacional')['rs_ano'], 1.0, 'nb07')
sub = a[(a['categoria_problema']=='Onde está meu pedido?') & (a['canal_entrada']!='ChatBot')]
checar('Impacto: economia ChatBot (R$/ano)',
       (sub['custo_operacional_ticket'].sum() - len(sub)*2.0)/3, imp('chatbot')['rs_ano'], 1.0, 'nb07')

# ruptura curva A (receita em risco anualizada)
sku_rec = v.groupby('sku_id')['receita_liquida'].sum().sort_values(ascending=False)
curva = pd.cut(sku_rec.cumsum()/sku_rec.sum(), bins=[-0.01,0.80,0.95,1.0], labels=['A','B','C'])
skus_a = curva[curva=='A'].index
crit = estoque[estoque['sku_id'].isin(skus_a) & estoque['status_disponibilidade'].isin(['Ruptura','Estoque Crítico'])]
rec_crit = sku_rec.loc[sku_rec.index.isin(crit['sku_id'])].sum()
checar('Impacto: ruptura curva A (R$/ano)', rec_crit*anu, imp('ruptura')['rs_ano'], 1.0, 'nb07')

# denominador da régua + coerência dos %
checar('Denominador da régua (margem realizada R$/ano)', realiz['margem_contribuicao'].sum()*anu, IMP['regua']['denominador_rs_ano'], 1.0, 'nb07')
pd.DataFrame(resultados).tail(7)

,métrica,recalculado,publicado,confere,fonte
9,Impacto: margem não realizada (R$/ano),2399375.41,2399375.41,SIM,nb07
10,Impacto: desconto (R$/ano),1531876.76,1531876.76,SIM,nb07
11,Impacto: marketplace frete (R$/ano),139417.93,139417.93,SIM,nb07
12,Impacto: custo falha operacional (R$/ano),106702.67,106702.67,SIM,nb07
13,Impacto: economia ChatBot (R$/ano),46043.33,46043.33,SIM,nb07
14,Impacto: ruptura curva A (R$/ano),2098192.64,2098192.64,SIM,nb07
15,Denominador da régua (margem realizada R$/ano),7212699.92,7212699.92,SIM,nb07


## Bloco D — Coerência estrutural do placar

O placar de vereditos e a contagem de hipóteses têm que fechar em 14.

In [5]:
placar = IMP['placar']; total = sum(placar.values())
print('Placar:', placar, '=> total', total)
n_hip = len(IMP['hipoteses'])
n_impacto = len(IMP['impactos']); n_sem = len(IMP['sem_rs'])
print(f'Hipóteses: {n_hip} | com R$: {n_impacto} | sem R$: {n_sem}')
checar('Placar soma 14 hipóteses', total, 14, 0, 'nb07')
checar('Hipóteses catalogadas', n_hip, 14, 0, 'nb07')

Placar: {'Refutada': 4, 'Confirmada': 5, 'Não testável': 3, 'Parcial': 1, 'Com ressalva': 1} => total 14
Hipóteses: 14 | com R$: 6 | sem R$: 8


True

## Bloco F — As métricas de medida corrigida

As quatro correções que reordenaram a prioridade são exatamente as que mais precisam de um
segundo par de olhos, porque cada uma muda a posição de uma frente no ranking. Todas são
recomputadas aqui por caminho próprio.

In [6]:
# --- identidades contábeis ---
checar('Identidade receita líquida (linhas fora)',
       float(((vf['receita_bruta'] - vf['desconto_reais'] - vf['receita_liquida']).abs() > 0.01).sum()),
       pub('identidade_linhas_fora_1centavo'), 0, 'nb06')

# --- decomposição da H9: partição disjunta ---
dev_m = v.loc[v['devolvido'], 'margem_contribuicao'].sum() * anu
can_m = v.loc[(~v['devolvido']) & (v['status_pagamento'] == 'Cancelado'), 'margem_contribuicao'].sum() * anu
agu_m = v.loc[(~v['devolvido']) & (v['status_pagamento'] == 'Aguardando'), 'margem_contribuicao'].sum() * anu
checar('H9: devolução (R$/ano)', dev_m, pub('h9_devolucao_margem_ano'), 1.0, 'nb06')
checar('H9: pagamento cancelado (R$/ano)', can_m, pub('h9_cancelado_margem_ano'), 1.0, 'nb06')
checar('H9: pagamento aguardando (R$/ano)', agu_m, pub('h9_aguardando_margem_ano'), 1.0, 'nb06')
checar('H9: partição fecha no gap total', dev_m + can_m + agu_m, pub('margem_gap_rs_ano'), 1.0, 'partição')

ender = ['Produto com defeito', 'Tamanho errado', 'Atraso na entrega']
pm = v.loc[v['devolvido']].groupby('motivo_devolucao')['margem_contribuicao'].sum() * anu
checar('H9: devolução endereçável (R$/ano)', float(pm[pm.index.isin(ender)].sum()),
       pub('h9_dev_enderecavel_ano'), 1.0, 'nb06')
checar('H9: devolução não endereçável (R$/ano)', float(pm[~pm.index.isin(ender)].sum()),
       pub('h9_dev_nao_enderecavel_ano'), 1.0, 'nb06')

# --- ruptura em margem própria dos SKUs (sem taxa média) ---
sku_mc_rea = realiz.groupby('sku_id')['margem_contribuicao'].sum()
skus_crit = set(crit['sku_id'])
checar('Ruptura: margem realizada dos SKUs (R$/ano)',
       float(sku_mc_rea.loc[sku_mc_rea.index.isin(skus_crit)].sum() * anu),
       pub('ruptura_curvaA_margem_realizada_ano'), 1.0, 'nb06')
so_rup = set(estoque.loc[estoque['status_disponibilidade'] == 'Ruptura', 'sku_id']) & set(skus_a)
checar('Ruptura: SKUs efetivamente em ruptura', float(len(so_rup)), pub('ruptura_em_ruptura_skus'), 0, 'nb06')
checar('Ruptura: perda corrente (R$/ano)',
       float(sku_mc_rea.loc[sku_mc_rea.index.isin(so_rup)].sum() * anu),
       pub('ruptura_em_ruptura_margem_realizada_ano'), 1.0, 'nb06')

# --- desconto: sobreposição medida e excedente por teto ---
d_nao = v.loc[~v.index.isin(realiz.index), 'desconto_reais'].sum()
checar('Desconto: sobreposto com a H9 (R$/ano)', d_nao * anu,
       pub('desconto_sobreposto_nao_realizado_ano'), 1.0, 'nb06')
checar('Desconto: incremento real sobre a H9 (R$/ano)', realiz['desconto_reais'].sum() * anu,
       pub('desconto_adicional_realizado_ano'), 1.0, 'nb06')
rz = realiz.copy()
rz['dp'] = np.where(rz['receita_bruta'] > 0, rz['desconto_reais'] / rz['receita_bruta'], 0.0)
for teto, chave in [(0.20, 'desconto_excedente_teto20_ano'), (0.15, 'desconto_excedente_teto15_ano')]:
    ac = rz[rz['dp'] > teto]
    checar('Desconto: excedente acima de {:.0%} (R$/ano)'.format(teto),
           float(((ac['dp'] - teto) * ac['receita_bruta']).sum() * anu), pub(chave), 1.0, 'nb06')

# --- primitivas de integridade das bases ---
mkt = carregar('Marketing')
checar('Marketing: receita declarada / real (x)', mkt['receita_gerada'].sum() / rb_tot,
       pub('mkt_receita_declarada_vs_real_x'), 0.05, 'nb06')
checar('Clientes: cobertura em Vendas (%)', 100 * v['customer_id'].nunique() / len(clientes),
       pub('clientes_cobertura_pct'), 0.02, 'nb06')
checar('Atendimento: frases distintas', float(a['texto_cliente'].nunique()),
       pub('atend_frases_distintas'), 0, 'nb06')
pd.DataFrame(resultados).tail(16)

,métrica,recalculado,publicado,confere,fonte
19,H9: devolução (R$/ano),1425548.37,1425548.37,SIM,nb06
20,H9: pagamento cancelado (R$/ano),653067.07,653067.07,SIM,nb06
21,H9: pagamento aguardando (R$/ano),320759.97,320759.97,SIM,nb06
22,H9: partição fecha no gap total,2399375.41,2399375.41,SIM,partição
23,H9: devolução endereçável (R$/ano),989664.15,989664.15,SIM,nb06
24,H9: devolução não endereçável (R$/ano),435884.22,435884.22,SIM,nb06
25,Ruptura: margem realizada dos SKUs (R$/ano),867128.17,867128.17,SIM,nb06
26,Ruptura: SKUs efetivamente em ruptura,54.00,54.00,SIM,nb06
27,Ruptura: perda corrente (R$/ano),120022.30,120022.30,SIM,nb06
28,Desconto: sobreposto com a H9 (R$/ano),379560.48,379560.48,SIM,nb06


## Bloco G — A camada de priorização e de solução (nb08)

Aqui o check não é sobre o cálculo interno do modelo — é sobre **rastreabilidade e invariantes**:

1. todo R$ que a priorização usa é exatamente o valor da chave canônica que ela declara ter usado;
2. cada iniciativa pertence a exatamente um módulo, e a soma bate;
3. **nenhum módulo opera num degrau de automação que o dado da sua decisão não sustenta** — é o
   invariante que o pilar de factibilidade representa, e o único que, se quebrado, faria a proposta
   prometer automação que a base não aguenta.

In [7]:
PRI = json.loads(Path('outputs/priorizacao.json').read_text(encoding='utf-8'))
MODS = PRI['modulos']
NIV = PRI['parametros']['niveis']
POR_ID = {i['id']: i for i in PRI['iniciativas']}

# 1. cada iniciativa com R$ aponta para uma chave canônica e o valor bate com ela
falhas = []
for ini in PRI['iniciativas']:
    ch = ini['chave_rs']
    if ch and ch != '-':
        if ch not in CANON:
            falhas.append((ini['id'], ch, 'chave inexistente'))
        elif abs(CANON[ch]['valor'] - ini['rs_ano']) > 0.01:
            falhas.append((ini['id'], ch, 'valor diverge da chave'))
    elif ini['rs_ano'] != 0:
        falhas.append((ini['id'], '-', 'R$ sem chave de origem'))
checar('Iniciativas: R$ rastreia até chave canônica', float(len(falhas)), 0, 0, 'nb08')
if falhas:
    print('FALHAS DE RASTREIO:', falhas)

# 2. módulos: soma fecha, cobertura total, sem dupla alocação
erros_soma = [m['nome'] for m in MODS
              if abs(sum(POR_ID[i]['rs_ano'] for i in m['inis']) - m['rs_ano']) > 0.01]
checar('Módulos: soma das iniciativas fecha', float(len(erros_soma)), 0, 0, 'nb08')
alocadas = [i for m in MODS for i in m['inis']]
checar('Toda iniciativa alocada a um módulo', float(len(set(alocadas))), float(len(PRI['iniciativas'])), 0, 'nb08')
checar('Nenhuma iniciativa em dois módulos', float(len(alocadas) - len(set(alocadas))), 0, 0, 'nb08')

# 3. INVARIANTE: o degrau em que o módulo opera tem de ser sustentado pelo dado da decisão
violacoes = []
for m in MODS:
    n = NIV[m['degrau']]
    if n['dado_min'] > m['dado']:
        violacoes.append((m['nome'], m['degrau'], 'dado {} < mínimo {}'.format(m['dado'], n['dado_min'])))
    if n['precisa_rotulo'] and not m['rotulo']:
        violacoes.append((m['nome'], m['degrau'], 'degrau exige histórico rotulado e o módulo não tem'))
checar('Nenhum módulo opera acima do que o dado sustenta', float(len(violacoes)), 0, 0, 'nb08')
if violacoes:
    print('VIOLAÇÕES DE DEGRAU:', violacoes)

# 4. o degrau operado nunca é mais ambicioso que o necessário
acima = [m['nome'] for m in MODS if NIV[m['degrau']]['ordem'] > NIV[m['precisa']]['ordem']]
checar('Nenhum módulo constrói acima da necessidade', float(len(acima)), 0, 0, 'nb08')

# 5. totais publicados batem com a soma das iniciativas
imediato = sum(i['rs_ano'] for i in PRI['iniciativas'] if i['classe'] == 'Ganho imediato')
checar('Total capturável no horizonte (R$/ano)', imediato, PRI['totais']['capturavel_horizonte_rs_ano'], 1.0, 'nb08')
checar('Total endereçável (R$/ano)', sum(i['rs_ano'] for i in PRI['iniciativas']),
       PRI['totais']['enderecavel_total_rs_ano'], 1.0, 'nb08')

# 6. as 14 hipóteses têm destino, e a régua é a mesma do nb07
hips = {i['hip'] for i in PRI['iniciativas']} | {d['hipotese'] for d in PRI['descarte']}
checar('As 14 hipóteses têm destino', float(len(hips)), 14.0, 0, 'nb08')
checar('Régua idêntica à do nb07', PRI['regua']['denominador_rs_ano'], IMP['regua']['denominador_rs_ano'], 0.01, 'nb08')

# 7. o vencedor publicado é de fato o primeiro do ranking
ranked = sorted([m for m in MODS if m['rank']], key=lambda m: m['rank'])
checar('Vencedor publicado é o 1º do ranking', 1.0 if ranked[0]['nome'] == PRI['vencedor']['modulo'] else 0.0, 1.0, 0, 'nb08')
checar('Cenários de sensibilidade executados', float(len(PRI['sensibilidade'])), 6.0, 0, 'nb08')

print('Vencedor: {} | degrau "{}" | DVF {} | estável na sensibilidade: {}'.format(
      PRI['vencedor']['modulo'], PRI['vencedor']['degrau'], PRI['vencedor']['dvf'],
      PRI['vencedor']['estavel_na_sensibilidade']))
pd.DataFrame(resultados).tail(11)

Vencedor: Recuperação de receita pós-venda | degrau "recomendacao" | DVF 8.67 | estável na sensibilidade: False


,métrica,recalculado,publicado,confere,fonte
36,Módulos: soma das iniciativas fecha,0.00,0.00,SIM,nb08
37,Toda iniciativa alocada a um módulo,13.00,13.00,SIM,nb08
38,Nenhuma iniciativa em dois módulos,0.00,0.00,SIM,nb08
39,Nenhum módulo opera acima do que o dado sustenta,0.00,0.00,SIM,nb08
40,Nenhum módulo constrói acima da necessidade,0.00,0.00,SIM,nb08
41,Total capturável no horizonte (R$/ano),1265013.63,1265013.63,SIM,nb08
42,Total endereçável (R$/ano),2666864.01,2666864.01,SIM,nb08
43,As 14 hipóteses têm destino,14.00,14.00,SIM,nb08
44,Régua idêntica à do nb07,7212699.92,7212699.92,SIM,nb08
45,Vencedor publicado é o 1º do ranking,1.00,1.00,SIM,nb08


## Bloco H — As dashboards não citam arquivo que não existe

Duas verificações que a versão anterior deste notebook não fazia — e é exatamente por isso
que referências a arquivos-fantasma sobreviveram numa entrega:

1. todo arquivo citado nas páginas HTML existe no disco;
2. nenhuma página menciona material de treinamento. As páginas são entregáveis de
   consultoria para o cliente; referência a enunciado ou a catálogo de curso não pertence ali.

In [8]:
import re

HTMLS = sorted(p for p in Path('.').glob('*.html') if not p.name.startswith('DESCONSIDERADO'))
print('Páginas auditadas:', ', '.join(p.name for p in HTMLS))

fantasmas, proibidos = [], []
TERMOS = ['bootcamp', 'enunciado', 'elogroup', 'material do curso']
for p in HTMLS:
    txt = p.read_text(encoding='utf-8')
    for arq in set(re.findall(r'[\w\-/]+\.(?:ipynb|json|py|md|csv)', txt)):
        if not (Path(arq).exists() or (Path('outputs') / Path(arq).name).exists()):
            fantasmas.append((p.name, arq))
    sem_tags = re.sub(r'<(script|style)[^>]*>.*?</\1>', '', txt, flags=re.S)
    for t in TERMOS:
        if t in sem_tags.lower():
            proibidos.append((p.name, t))

checar('Arquivos citados nas páginas existem', float(len(fantasmas)), 0, 0, 'dashboards')
checar('Páginas sem referência a material de treinamento', float(len(proibidos)), 0, 0, 'dashboards')
if fantasmas:
    print('ARQUIVOS-FANTASMA:', fantasmas)
if proibidos:
    print('REFERÊNCIAS PROIBIDAS:', proibidos)
if not fantasmas and not proibidos:
    print('OK: nenhuma citação quebrada, nenhuma referência indevida.')

Páginas auditadas: diagnostico_vertice_v3.html, proposta_solucao_vertice_v3.html
OK: nenhuma citação quebrada, nenhuma referência indevida.


## Bloco E — Veredito final

In [9]:
tab = pd.DataFrame(resultados)
n_ok = (tab['confere']=='SIM').sum(); n = len(tab)
print(f"{n_ok} de {n} métricas conferem (dentro da tolerância) contra os JSONs canônicos.")
if n_ok < n:
    print('\nDIVERGÊNCIAS:')
    print(tab[tab['confere']!='SIM'].to_string(index=False))
else:
    print('Nenhuma divergência: os notebooks 06, 07 e 08 estão consistentes com a base bruta,')
    print('e as dashboards não citam arquivo inexistente nem material de treinamento.')
tab

49 de 49 métricas conferem (dentro da tolerância) contra os JSONs canônicos.
Nenhuma divergência: os notebooks 06, 07 e 08 estão consistentes com a base bruta,
e as dashboards não citam arquivo inexistente nem material de treinamento.


,métrica,recalculado,publicado,confere,fonte
0,Pedidos válidos,27758.00,27758.00,SIM,nb06
1,Margem contábil %,54.37,54.37,SIM,nb06
2,Margem realizada %,40.80,40.80,SIM,nb06
3,Gap de margem 13m (R$),2563716.19,2563716.19,SIM,nb06
4,Desconto total 13m (R$),1636799.83,1636799.83,SIM,nb06
5,Desconto % receita bruta,7.97,7.97,SIM,nb06
6,Marketplace gap (R$ 13m),148967.10,148967.10,SIM,nb06
7,Marketplace frete % agregado,4.93,4.93,SIM,nb06
8,r² desconto×margem,0.29,0.29,SIM,nb06
9,Impacto: margem não realizada (R$/ano),2399375.41,2399375.41,SIM,nb07
